# CatBoost Model - Riccardo
### NYC 311 Service Requests
## Predicting if a ticket is closed within 24 hours

This notebook builds a **CatBoost classifier** for the project.

**Target variable:**
- `Y = 1` if the service request is closed within 24 hours
- `Y = 0` if the service request takes longer than 24 hours or is still open after 24 hours

We use Matteo's shared cleaning file, `Matteo/DS_processing.py`, so the data preparation is consistent with the rest of the team.

## 1. Imports and Paths

First we set the path to Riccardo's folder. This makes the notebook work both when it is run from the repository root and when it is run from inside the `notebooks` folder.

If CatBoost is missing in your notebook kernel, run this once:

```python
%pip install -r ../requirements.txt
```

In [ ]:
from pathlib import Path
import sys

notebook_cwd = Path.cwd()

if notebook_cwd.name == "notebooks":
    base_dir = notebook_cwd.parent
elif (notebook_cwd / "Riccardo").exists():
    base_dir = notebook_cwd / "Riccardo"
else:
    base_dir = notebook_cwd

sys.path.insert(0, str(base_dir))
print(f"Using Riccardo folder: {base_dir}")

In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from catboost_model import (
    CATEGORICAL_COLS,
    FEATURE_COLS,
    NUMERICAL_COLS,
    Process_test_DS,
    Process_train_DS,
    RANDOM_STATE,
    SUBMISSION_DIR,
    best_accuracy_threshold,
    build_feature_frame,
    build_model,
    load_raw_data,
    print_dataset_checks,
    save_feature_importance,
)

print("All imports successful!")

## 2. Load the Raw Data

We load:
- `train.csv`, which contains the dates needed to create the target variable
- `test.csv`, which has no `Closed Date` and is used only at the final prediction step
- `submission.csv`, which gives the required final CSV format

The raw CSV files are kept locally and are not pushed to GitHub.

In [ ]:
train_raw, test_raw, sample_submission = load_raw_data()

print(f"Training set: {train_raw.shape[0]} rows, {train_raw.shape[1]} columns")
print(f"Test set:     {test_raw.shape[0]} rows, {test_raw.shape[1]} columns")
print(f"Submission:   {sample_submission.shape[0]} rows, {sample_submission.shape[1]} columns")

## 3. Process the Data

We use Matteo's `Process_train_DS` and `Process_test_DS` functions.

These functions:
- group detailed complaint types into broader problem categories
- group detailed location types into broader location categories
- create binary indicators such as `Is_Landmark` and `Is_Taxi`
- extract time features from `Created Date`
- create the target variable `Y` for the training set
- drop columns that are redundant, too granular, or would create leakage

In [ ]:
train_df = Process_train_DS(train_raw)
test_df = Process_test_DS(test_raw)

print("Training set after processing:")
print(train_df.shape)
train_df.head(3)

## 4. Quick Data Checks

Before modeling, we check that the processed data has no null values and that the target variable has the expected class balance.

This follows Matteo's instruction to run `info()` or a similar check before training the model.

In [ ]:
print_dataset_checks(train_df, test_df)

## 5. Define Features and Target

I separate the data into:
- **X**: the input features that the model can use
- **y**: the target variable we want to predict

CatBoost can handle categorical features directly, so we do not need one-hot encoding or manual target encoding.

First I tested CatBoost using only the features returned by Matteo's `DS_processing.py`; that baseline reached about 87.97% validation accuracy.

Then I added safe raw features that are available in both train and test, such as the original complaint type, complaint detail, location type, city/community information, coordinates, street name, BBL, and extra `Created Date` features. With those features, validation accuracy increased to about 90.6%.

In [ ]:
X = build_feature_frame(train_df, train_raw)
y = train_df["Y"].astype(int)
X_test = build_feature_frame(test_df, test_raw)

print("Categorical features:")
print(CATEGORICAL_COLS)

print("\nNumerical features:")
print(NUMERICAL_COLS)

print("\nAll model features:")
print(FEATURE_COLS)

## 6. Split into Training and Validation Sets

We split the training data into:
- **80% train**: the model learns from this part
- **20% validation**: we use this to estimate performance on unseen data

We use `stratify=y` so that both splits keep the same proportion of `Y = 1` and `Y = 0`.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Training:   {X_train.shape[0]} rows")
print(f"Validation: {X_val.shape[0]} rows")
print(f"\nY proportion in train:      {y_train.mean():.3f}")
print(f"Y proportion in validation: {y_val.mean():.3f}")

## 7. Build the CatBoost Model

CatBoost is a gradient boosting model that is especially useful for datasets with categorical variables.

Why CatBoost fits this project:
- several important variables are categorical, such as `Agency`, `Incident Zip`, and `Problem_Grouped`
- it handles categorical variables natively
- it avoids arbitrary label ordering from label encoding
- it reduces the need to manually implement leakage-safe target encoding
- early stopping helps control overfitting

In [ ]:
train_pool = Pool(X_train, y_train, cat_features=CATEGORICAL_COLS)
val_pool = Pool(X_val, y_val, cat_features=CATEGORICAL_COLS)

model = build_model(
    iterations=2500,
    depth=8,
    learning_rate=0.04,
    l2_leaf_reg=8.0,
)

print("Model ready!")

## 8. Train the Model

We train on the training split and use the validation split for early stopping.

The model keeps the best iteration according to validation accuracy.

In [ ]:
model.fit(train_pool, eval_set=val_pool, use_best_model=True)

print("Model trained!")
print(f"Best iteration: {model.get_best_iteration()}")

## 9. Evaluate on the Validation Set

Accuracy is the official evaluation metric, so we focus on validation accuracy.

I also check whether a threshold other than `0.5` improves accuracy. In the best experiment, the optimal threshold was close to `0.5`.

In [ ]:
val_probabilities = model.predict_proba(val_pool)[:, 1]
default_predictions = (val_probabilities >= 0.5).astype(int)

default_accuracy = accuracy_score(y_val, default_predictions)
best_threshold, best_accuracy = best_accuracy_threshold(y_val, val_probabilities)
best_predictions = (val_probabilities >= best_threshold).astype(int)

print(f"Default threshold accuracy: {default_accuracy:.5f}")
print(f"Best validation threshold: {best_threshold:.3f}")
print(f"Best validation accuracy:  {best_accuracy:.5f}")

print("\nDetailed report:")
print(classification_report(y_val, best_predictions, target_names=["Not closed in 24h", "Closed in 24h"]))

print("Confusion matrix:")
print(confusion_matrix(y_val, best_predictions))

## 10. Feature Importance

Feature importance helps us explain what the model relies on most.

In the stronger run, the model can use both Matteo's grouped variables and the original detailed complaint/location variables. This gives us a more accurate model while still keeping the feature choices explainable.

In [ ]:
save_feature_importance(model, train_pool)

## 11. Robustness and Overfitting Checks

Because some added variables are location-specific, I checked whether the model was simply memorizing locations.

The most important checks were:

| Check | Accuracy / Gap |
| --- | ---: |
| Baseline using only `DS_processing.py` features | about 87.97% |
| Added raw detail/location/date features | about 90.6% |
| 5-fold CV tuned accuracy | about 90.67% +/- 0.17 pp |
| Conservative model without `Street Name`, `BBL`, coordinates | about 90.27% CV validation accuracy |
| Train-validation gap, conservative model | about 2.31 pp |
| Train-validation gap, full model | about 3.68 pp |

Interpretation: the main improvement comes from raw descriptive fields such as complaint detail and location type, not only from highly specific location IDs. The full model has the highest validation accuracy, while the conservative model is slightly easier to defend because its train-validation gap is smaller.

## 12. Train Final Model and Predict on Test Set

After validating the model, we train one final CatBoost model on the full processed training set.

Then we predict on `test.csv`. This is the only time the test set is used.

In [ ]:
best_iteration = model.get_best_iteration()
final_iterations = int(best_iteration) + 1 if best_iteration is not None and best_iteration > 0 else 2500

final_model = build_model(
    iterations=final_iterations,
    depth=8,
    learning_rate=0.04,
    l2_leaf_reg=8.0,
)

full_train_pool = Pool(X, y, cat_features=CATEGORICAL_COLS)
test_pool = Pool(X_test, cat_features=CATEGORICAL_COLS)

final_model.fit(full_train_pool)

test_probabilities = final_model.predict_proba(test_pool)[:, 1]
test_predictions = (test_probabilities >= best_threshold).astype(int)

print(f"Predictions made: {len(test_predictions)} rows")
print(f"Predicted Y=1: {test_predictions.sum()} ({test_predictions.mean() * 100:.1f}%)")
print(f"Predicted Y=0: {(test_predictions == 0).sum()} ({(1 - test_predictions.mean()) * 100:.1f}%)")

## 13. Save the Submission File

The final CSV must match the format of `submission.csv` exactly.

The generated file is saved locally in `Riccardo/submissions/` and is ignored by Git, so it will not be pushed to GitHub.

In [ ]:
submission = sample_submission.copy()
submission["prediction"] = test_predictions

SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
submission_path = SUBMISSION_DIR / "catboost_submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print("\nFirst few rows:")
submission.head()